<a href="https://colab.research.google.com/github/yajima-yasutoshi/Model2026/blob/main/20260715/k%E3%82%BB%E3%83%B3%E3%82%BF%E3%83%BC%E5%95%8F%E9%A1%8C%E3%81%AE%E6%BC%94%E7%BF%92%E5%95%8F%E9%A1%8C%E3%81%AE%E8%A7%A3%E8%AA%AC%E3%81%A8%E8%A7%A3%E7%AD%94.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# k-センター問題

### 準備: ライブラリのインストールとインポート


In [ ]:
%%capture
# python-mip ライブラリをインストールする
!pip install mip

# Matplotlibで日本語を表示するためのライブラリをインストール
!pip install japanize-matplotlib

# 必要なライブラリをインポートする
import mip
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib # ← 日本語表示のためにインポート
import random
from math import sqrt
from itertools import product
from scipy.spatial.distance import cdist # 距離行列計算用

---
## 演習問題 1

### 1. 問題
「問題データの設定」セクションで生成された15地点のデータと $k=5$ を用いてk-センター問題を解き、最適値（最小化された最大距離）を求めよ。

### 2. 数理モデルの定式化
本問題は、講義で示された標準的なk-センター問題の数理モデルを用いて定式化される。目的は、全需要地点とその割り当て先施設との距離の最大値 $Z$ を最小化することである。

**目的関数:**
$$\min Z$$

**制約条件:**
1.  **割り当て制約:** 各需要地点 $i$ は、ただ一つの施設 $j$ に割り当てられる。
$$\sum_{j \in J} x_{ij} = 1 \quad (\forall i \in I)$$
2.  **施設開設連動制約:** 需要地点 $i$ は、施設が開設された場所 $j$ にのみ割り当て可能である。
$$x_{ij} \le y_j \quad (\forall i \in I, \forall j \in J)$$
3.  **施設数制約:** 開設される施設の総数は $k$ 個である。
$$\sum_{j \in J} y_j = k$$
4.  **最大距離定義制約:** 各需要地点 $i$ から割り当てられた施設への距離は、$Z$ を超えてはならない。
$$\sum_{j \in J} d_{ij} x_{ij} \le Z \quad (\forall i \in I)$$

ここで、$k=5$ として設定する。

### 3. Python (MIP) による実装

In [ ]:
# --- 演習問題1 データ設定 ---
num_locations_kc = 15
k_centers_to_locate = 5 # 開設する施設数を5に設定
random.seed(50) # 講義の例と同じ地点データを生成
locations_coords_kc = np.array([(random.uniform(0, 100), random.uniform(0, 100)) for _ in range(num_locations_kc)])
dist_matrix_kc = cdist(locations_coords_kc, locations_coords_kc, 'euclidean')

# --- モデルの作成 ---
model_ex1 = mip.Model(name="k_center_ex1", sense=mip.MINIMIZE)

# インデックスの集合
I_nodes = range(num_locations_kc)
J_nodes = range(num_locations_kc)

# --- 変数の定義 ---
# y_j: 施設候補地jに施設を開設するなら1
y = [model_ex1.add_var(var_type=mip.BINARY, name=f"y_{j}") for j in J_nodes]
# x_ij: 需要地点iを施設jに割り当てるなら1
x = {(i, j): model_ex1.add_var(var_type=mip.BINARY, name=f"x_{i}_{j}") for i in I_nodes for j in J_nodes}
# Z: 最大割り当て距離 (連続変数)
Z = model_ex1.add_var(name="Z_max_distance", lb=0.0)

# --- 目的関数の設定 ---
model_ex1.objective = Z

# --- 制約条件の追加 ---
# 制約1: 各需要地点は、いずれか一つの施設に割り当てられる
for i in I_nodes:
    model_ex1 += mip.xsum(x[(i, j)] for j in J_nodes) == 1, f"assign_cust_{i}"

# 制約2: 需要地点は、施設が開設されている場合にのみ割り当て可能
for i in I_nodes:
    for j in J_nodes:
        model_ex1 += x[(i, j)] <= y[j], f"link_assign_open_{i}_{j}"

# 制약3: 開設する施設の総数はk個
model_ex1 += mip.xsum(y[j] for j in J_nodes) == k_centers_to_locate, f"num_facilities"

# 制約4: 最大距離の定義
for i in I_nodes:
    model_ex1 += mip.xsum(dist_matrix_kc[i, j] * x[(i, j)] for j in J_nodes) <= Z, f"max_dist_def_{i}"

# --- 問題の求解 ---
status = model_ex1.optimize()

# --- 結果の表示 ---
if status == mip.OptimizationStatus.OPTIMAL:
    print("最適解が見つかりました！")
    minimized_max_dist = Z.x
    print(f"最小化された最大割り当て距離 (Z): {minimized_max_dist:.4f}")
    opened_centers = [j for j in J_nodes if y[j].x >= 0.99]
    print(f"開設されたセンターの位置 (インデックス): {opened_centers}")
elif status == mip.OptimizationStatus.FEASIBLE:
    print("実行可能解が見つかりました(最適ではない可能性あり)。")
    minimized_max_dist = Z.x
    print(f"最小化された最大割り当て距離 (Z): {minimized_max_dist:.4f}")
else:
    print(f"解が見つかりませんでした。最適化ステータス: {status}")

最適解が見つかりました！
最小化された最大割り当て距離 (Z): 28.2654
開設されたセンターの位置 (インデックス): [0, 3, 5, 6, 11]


---
## 演習問題 2

### 1. 問題
演習問題1と同じ15地点のデータを用い、開設するセンター数 $k$ を2に変更しk-センター問題を解き、最適値（最小化された最大距離）を求めよ。

### 2. 数理モデルの定式化
数理モデルの構造は演習問題1と同様である。変更点は、施設数制約におけるパラメータ $k$ の値を2とすることのみである。
$$\sum_{j \in J} y_j = 2$$

### 3. Python (MIP) による実装

In [ ]:
# --- 演習問題2 データ設定 ---
num_locations_kc = 15
k_centers_to_locate = 2 # 開設する施設数を2に変更
random.seed(50) # 講義の例と同じ地点データを生成
locations_coords_kc = np.array([(random.uniform(0, 100), random.uniform(0, 100)) for _ in range(num_locations_kc)])
dist_matrix_kc = cdist(locations_coords_kc, locations_coords_kc, 'euclidean')

# --- モデルの作成 ---
model_ex2 = mip.Model(name="k_center_ex2", sense=mip.MINIMIZE)

# インデックスの集合
I_nodes = range(num_locations_kc)
J_nodes = range(num_locations_kc)

# --- 変数の定義 ---
y = [model_ex2.add_var(var_type=mip.BINARY, name=f"y_{j}") for j in J_nodes]
x = {(i, j): model_ex2.add_var(var_type=mip.BINARY, name=f"x_{i}_{j}") for i in I_nodes for j in J_nodes}
Z = model_ex2.add_var(name="Z_max_distance", lb=0.0)

# --- 目的関数の設定 ---
model_ex2.objective = Z

# --- 制約条件の追加 ---
# 制約1
for i in I_nodes:
    model_ex2 += mip.xsum(x[(i, j)] for j in J_nodes) == 1
# 制約2
for i in I_nodes:
    for j in J_nodes:
        model_ex2 += x[(i, j)] <= y[j]
# 制約3
model_ex2 += mip.xsum(y[j] for j in J_nodes) == k_centers_to_locate
# 制約4
for i in I_nodes:
    model_ex2 += mip.xsum(dist_matrix_kc[i, j] * x[(i, j)] for j in J_nodes) <= Z

# --- 問題の求解 ---
status = model_ex2.optimize()

# --- 結果の表示 ---
if status == mip.OptimizationStatus.OPTIMAL:
    print("最適解が見つかりました！")
    minimized_max_dist = Z.x
    print(f"最小化された最大割り当て距離 (Z): {minimized_max_dist:.4f}")
    opened_centers = [j for j in J_nodes if y[j].x >= 0.99]
    print(f"開設されたセンターの位置 (インデックス): {opened_centers}")
else:
    print(f"解が見つかりませんでした。最適化ステータス: {status}")

最適解が見つかりました！
最小化された最大割り当て距離 (Z): 48.5108
開設されたセンターの位置 (インデックス): [0, 2]


---
## 演習問題 3

### 1. 問題
距離行列 $d_{ij}$ をユークリッド距離ではなく、マンハッタン距離に変更して、$k=3$ でk-センター問題を解き、最適値（最小化された最大距離）を求めよ。

### 2. 数理モデルの定式化
数理モデルは標準形と同じであるが、パラメータ $d_{ij}$ の計算方法が異なる。
$$d_{ij} = |u_i - u_j| + |v_i - v_j|$$
ここで、$(u_i, v_i)$ は地点 $i$ の座標を示す。また、$k=3$ とする。

### 3. Python (MIP) による実装

In [ ]:
# --- 演習問題3 データ設定 ---
num_locations_kc = 15
k_centers_to_locate = 3 # 開設する施設数を3に設定
random.seed(50) # 講義の例と同じ地点データを生成
locations_coords_kc = np.array([(random.uniform(0, 100), random.uniform(0, 100)) for _ in range(num_locations_kc)])
# 距離行列をマンハッタン距離で計算
dist_matrix_kc = cdist(locations_coords_kc, locations_coords_kc, 'cityblock')

# --- モデルの作成 ---
model_ex3 = mip.Model(name="k_center_ex3", sense=mip.MINIMIZE)

# インデックスの集合
I_nodes = range(num_locations_kc)
J_nodes = range(num_locations_kc)

# --- 変数の定義 ---
y = [model_ex3.add_var(var_type=mip.BINARY, name=f"y_{j}") for j in J_nodes]
x = {(i, j): model_ex3.add_var(var_type=mip.BINARY, name=f"x_{i}_{j}") for i in I_nodes for j in J_nodes}
Z = model_ex3.add_var(name="Z_max_distance", lb=0.0)

# --- 目的関数の設定 ---
model_ex3.objective = Z

# --- 制約条件の追加 ---
# 制約1
for i in I_nodes:
    model_ex3 += mip.xsum(x[(i, j)] for j in J_nodes) == 1
# 制約2
for i in I_nodes:
    for j in J_nodes:
        model_ex3 += x[(i, j)] <= y[j]
# 制約3
model_ex3 += mip.xsum(y[j] for j in J_nodes) == k_centers_to_locate
# 制約4
for i in I_nodes:
    model_ex3 += mip.xsum(dist_matrix_kc[i, j] * x[(i, j)] for j in J_nodes) <= Z

# --- 問題の求解 ---
status = model_ex3.optimize()

# --- 結果の表示 ---
if status == mip.OptimizationStatus.OPTIMAL:
    print("最適解が見つかりました！")
    minimized_max_dist = Z.x
    print(f"最小化された最大割り当て距離 (Z): {minimized_max_dist:.4f}")
    opened_centers = [j for j in J_nodes if y[j].x >= 0.99]
    print(f"開設されたセンターの位置 (インデックス): {opened_centers}")
else:
    print(f"解が見つかりませんでした。最適化ステータス: {status}")

最適解が見つかりました！
最小化された最大割り当て距離 (Z): 50.9679
開設されたセンターの位置 (インデックス): [2, 12, 13]


---
## 演習問題 4

### 1. 問題
「問題データの設定」セクションで生成された15地点のデータ（$k=3$）において、地点インデックス0に必ずセンターを設置するという制約を加え、k-センター問題を解き、最適値（最小化された最大距離）を求めよ。

### 2. 数理モデルの定式化
標準のk-センターモデルに、特定の地点（インデックス0）に施設を必ず開設するという追加の制約を加える。これは、決定変数 $y_0$ の値を1に固定することで表現できる。

**追加制約:**
$$y_0 = 1$$

この制約により、最適化ソルバーは $y_0=1$ を前提として、
残りの $2$ 個のセンターを最適に配置することになる。

### 3. Python (MIP) による実装

In [ ]:
# --- 演習問題4 データ設定 ---
num_locations_kc = 15
k_centers_to_locate = 3 # 開設する施設数
random.seed(50) # 講義の例と同じ地点データを生成
locations_coords_kc = np.array([(random.uniform(0, 100), random.uniform(0, 100)) for _ in range(num_locations_kc)])
dist_matrix_kc = cdist(locations_coords_kc, locations_coords_kc, 'euclidean')

# --- モデルの作成 ---
model_ex4 = mip.Model(name="k_center_ex4", sense=mip.MINIMIZE)

# インデックスの集合
I_nodes = range(num_locations_kc)
J_nodes = range(num_locations_kc)

# --- 変数の定義 ---
y = [model_ex4.add_var(var_type=mip.BINARY, name=f"y_{j}") for j in J_nodes]
x = {(i, j): model_ex4.add_var(var_type=mip.BINARY, name=f"x_{i}_{j}") for i in I_nodes for j in J_nodes}
Z = model_ex4.add_var(name="Z_max_distance", lb=0.0)

# --- 目的関数の設定 ---
model_ex4.objective = Z

# --- 制約条件の追加 ---
# 制約1
for i in I_nodes:
    model_ex4 += mip.xsum(x[(i, j)] for j in J_nodes) == 1
# 制約2
for i in I_nodes:
    for j in J_nodes:
        model_ex4 += x[(i, j)] <= y[j]
# 制約3
model_ex4 += mip.xsum(y[j] for j in J_nodes) == k_centers_to_locate

# 制約4
for i in I_nodes:
    model_ex4 += mip.xsum(dist_matrix_kc[i, j] * x[(i, j)] for j in J_nodes) <= Z

# ★★★ 演習問題4の追加制約 ★★★
# 地点0に必ずセンターを設置する
model_ex4 += y[0] == 1, "force_open_center_0"

# --- 問題の求解 ---
status = model_ex4.optimize()

# --- 結果の表示 ---
if status == mip.OptimizationStatus.OPTIMAL:
    print("最適解が見つかりました！")
    minimized_max_dist = Z.x
    print(f"最小化された最大割り当て距離 (Z): {minimized_max_dist:.4f}")
    opened_centers = [j for j in J_nodes if y[j].x >= 0.99]
    print(f"開設されたセンターの位置 (インデックス): {opened_centers}")
else:
    print(f"解が見つかりませんでした。最適化ステータス: {status}")

最適解が見つかりました！
最小化された最大割り当て距離 (Z): 39.4265
開設されたセンターの位置 (インデックス): [0, 8, 12]
